In [1]:
import os
import pandas as pd
import geopandas as gpd
import csv
import shutil

## RiverTile node time series csv creation from shapefile tiles

### Pull many nodes by loading a csv with nodes of interest:

In [ ]:
# Load the CSV file with column 'Node_ID' with nodes of interest
# df = pd.read_csv('/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/PTs/SWOTCalVal_YR_node_id.csv') # PT nodes SWORD v16
df = pd.read_csv('/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/PTs/SWOTCalVal_YR_KEY_20240704_20240826_v17b.csv') # PT nodes SWORD v17b
# df = pd.read_csv('/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/YR_domain_v16.csv') # GNSS nodes SWORD v16
# df = pd.read_csv('/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/YR_domain_v17b.csv') # GNSS nodes SWORD v17b

# Remove duplicate Node_ID values and convert to string list
# node_ids = df.drop_duplicates(subset=['Node_ID'])['Node_ID'].astype(str).tolist()
# v17b field is 'v17b_node_id'
node_ids = df.drop_duplicates(subset=['v17b_node_id'])['v17b_node_id'].astype(str).tolist()

print(node_ids)

['81270500160011', '81270500160551', '81270500150591', '81270500150011', '81270500170011', '81270500160271', '81270500160341', '81270500170521', '81270500160171', '81270100060501', '81270100060011', '81270100050901', '81270100050471', '81270100050181', '81270100050011', '81270100040601', '81270100040011', '81270100030591', '81250800020011', '81250800020781', '81250800030011', '81250800030181', '81250800030301', '81250800030551', '81250800040801', '81250800030711', '81260300040011', '81260300040021', '81260300040481', '81260300050031', '81260300050301', '81260300050391', '81260300050481', '81260300180021', '81260300060011', '81260300190011', '81260300190151', '81260300190351', '81260300190471', '81260300200011', '81260300210061', '81260300200361', '81260300060481', '81260300070011', '81260300150941', '81260300160011', '81260300160901', '81260300170011', '81260300170151', '81260300170411', '81260300170601', '81260300170761', '81260500010011', '81260400010271', '81260400010481', '81260400

In [49]:
# SWOT RiverTile folder and shapefile list

# Change directory based on SWORD version
# swot_rivertile_folder = "/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/SWOT/node/RiverTile_v16"
swot_rivertile_folder = "/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/SWOT/node/RiverTile_v17b"

SWOTshapefiles = [f for f in os.listdir(swot_rivertile_folder) if f.startswith("SWOT_L2_HR_RiverTile_Node") and f.endswith(".shp")]
print(len(SWOTshapefiles))

# Set project CRS (EPSG:32606 for Yukon Flats)
YR_crs = "epsg:32606"

307


Field versions currently selected to pull:
node_id,reach_id,time,time_tai,time_str,lat,lon,wse,wse_u,wse_r_u,width,width_u,area_total,area_tot_u,area_detct,
area_det_u,area_wse,layovr_val,node_dist,xtrk_dist,node_q,node_q_b,dark_frac,n_good_pix,rdr_sig0,xovr_cal_q,p_dist_out

In [50]:
# Output CSV for node timeseries
summary_csv_path = os.path.join(swot_rivertile_folder, "RiverTile_pt_node_timeseries_v17b.csv") #change filename as needed here!
fieldnames = [
    'SWOTFileName', 'node_id', 'reach_id', 'time', 'time_tai', 'time_str',
    'lat', 'lon', 'wse', 'wse_u', 'wse_r_u', 'width', 'width_u',
    'area_total', 'area_tot_u', 'area_detct', 'area_det_u', 'area_wse',
    'layovr_val', 'node_dist', 'xtrk_dist', 'node_q', 'node_q_b',
    'dark_frac', 'n_good_pix', 'rdr_sig0', 'xovr_cal_q', 'p_dist_out',
]
# need to add additional fields in csv here and at the bottom of the loop in next chunk

In [ ]:
# iterate through all SWOT shapefiles to pull fields for each node of interest and save to output csv
with open(summary_csv_path, mode='w', newline='') as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()

    # Iterate through all SWOT shapefiles
    for SWOTshapefile in SWOTshapefiles:
        # Extract filename info
        SWOTfilename = os.path.basename(SWOTshapefile)[26:70] #gets cycle ID, tile ID, and start/end datetime

        # Load the SWOT shapefile & set the CRS
        shapefile_path = os.path.join(swot_rivertile_folder, SWOTshapefile)
        swot_rivertile = gpd.read_file(shapefile_path).to_crs(YR_crs)

        # Just process rows matching Node_IDs of interest
        swot_rivertile['node_id'] = swot_rivertile['node_id'].astype(str)
        matching_nodes = swot_rivertile[swot_rivertile['node_id'].isin(node_ids)]

        # pull fields of interest to write to csv
        for _, row in matching_nodes.iterrows():
            print(f"Processing {SWOTfilename} | node_id {row['node_id']}")

            writer.writerow({
                'SWOTFileName': SWOTfilename,
                'node_id': row['node_id'],
                'reach_id': row['reach_id'],
                'time': row['time'],
                'time_tai': row['time_tai'],
                'time_str': row['time_str'],
                'lat': row['lat'],
                'lon': row['lon'],
                'wse': row['wse'],
                'wse_u': row['wse_u'],
                'wse_r_u': row['wse_r_u'],
                'width': row['width'],
                'width_u': row['width_u'],
                'area_total': row['area_total'],
                'area_tot_u': row['area_tot_u'],
                'area_detct': row['area_detct'],
                'area_det_u': row['area_det_u'],
                'area_wse': row['area_wse'],
                'layovr_val': row['layovr_val'],
                'node_dist': row['node_dist'],
                'xtrk_dist': row['xtrk_dist'],
                'node_q': row['node_q'],
                'node_q_b': row['node_q_b'],
                'dark_frac': row['dark_frac'],
                'n_good_pix': row['n_good_pix'],
                'rdr_sig0': row['rdr_sig0'],
                'xovr_cal_q': row['xovr_cal_q'],
                'p_dist_out': row['p_dist_out'],
            })


Processing 018_181_273R_20240716T092601_20240716T092612 | node_id 81270100060011
Processing 018_181_273R_20240716T092601_20240716T092612 | node_id 81270100060501
Processing 020_431_272R_20240905T011747_20240905T011758 | node_id 81270500150011
Processing 020_431_272R_20240905T011747_20240905T011758 | node_id 81270500150591
Processing 020_431_272R_20240905T011747_20240905T011758 | node_id 81270500160011
Processing 020_431_272R_20240905T011747_20240905T011758 | node_id 81270500160171
Processing 020_431_272R_20240905T011747_20240905T011758 | node_id 81270500160271
Processing 020_431_272R_20240905T011747_20240905T011758 | node_id 81270500160341
Processing 020_431_272R_20240905T011747_20240905T011758 | node_id 81270500160551
Processing 020_431_272R_20240905T011747_20240905T011758 | node_id 81270500170011
Processing 020_431_272R_20240905T011747_20240905T011758 | node_id 81270500170521
Processing 018_153_275L_20240715T092550_20240715T092601 | node_id 81260300170011
Processing 018_153_275L_2024

### Move RiverTile node data to the same directories based on SWORD version

In [ ]:
# Root directory from JPL containing all the subdirectories
root_dir = "/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/SWOT/JPL_delivery/reprocessed_rivertiles_asdelivered_vD_YF24_20250513"

# Destination folders for shapefiles ordered by SWORD version
# dest_v16 = "/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/SWOT/node/RiverTile_v16"
# dest_v17b = "/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/SWOT/node/RiverTile_v17b"

# PIXCVecRiver
dest_v16 = "/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/SWOT/pixvec/for_orthos/PIXCVecRiver_v16"
dest_v17b = "/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/SWOT/pixvec/for_orthos/PIXCVecRiver_v17b"

# Ensure destination directories exist
os.makedirs(dest_v16, exist_ok=True)
os.makedirs(dest_v17b, exist_ok=True)

# Shapefile extensions to copy
# file_extensions = {'.shp', '.cpg', '.dbf', '.prj', '.xml', '.shx'}

# PIXCVecRiver
file_extensions = {'.nc'}

In [ ]:
# Move the files!
# Walk through all subdirectories
for dirpath, dirnames, filenames in os.walk(root_dir):
    if dirpath.endswith("v16_v1.4.1_250429") or dirpath.endswith("v17b_v1.4.1_250429"):
        # Dictate destination based on folder name
        if dirpath.endswith("v16_v1.4.1_250429"):
            dest_dir = dest_v16
        else:
            dest_dir = dest_v17b

        # Copy all the shapefile components / .nc files for PIXCVecRiver
        for file in filenames:
            if os.path.splitext(file)[1].lower() in file_extensions:
                src_file = os.path.join(dirpath, file)
                dest_file = os.path.join(dest_dir, file)
                print(f"Copying {src_file} -> {dest_file}")
                shutil.copy2(src_file, dest_file)


Copying /Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/SWOT/JPL_delivery/reprocessed_rivertiles_asdelivered_vD_YF24_20250513/024_034L/020/SWOT_L1B_HR_SLC_020_024_034L/asdelivered_v1.4.1/SWOT_L2_HR_PIXC_020_024_034L/asdelivered_v1.4.2/SWOT_L2_HR_RiverTile_020_024_034L/SWOT_L2_HR_RiverTile_020_024_034L_asdelivered_nom_v17b_v1.4.1_250429/SWOT_L2_HR_PIXCVecRiver_020_024_034L_20240821T113904_20240821T113915_DevPID0_01.nc -> /Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/SWOT/pixvec/for_orthos/PIXCVecRiver_v17b/SWOT_L2_HR_PIXCVecRiver_020_024_034L_20240821T113904_20240821T113915_DevPID0_01.nc
Copying /Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/SWOT/JPL_delivery/reprocessed_rivertiles_asdelivered_vD_YF24_20250513/024_034L/020/SWOT_L1B_HR_SLC_020_024_034L/asdelivered_v1.4.1/SWOT_L2_HR_PIXC_020_024_034L/asdelivered_v1.4.2/SWOT_L2_HR_RiverTile_020_024_034L/SWOT_L2_HR_RiverTile_020_024_034L_asdelivered_nom_v17b_v1.4.1_250429/SWOT_L2_HR_RiverTile_020

## RiverTile reach time series csv creation from shapefile tiles

### Pull many reaches by loading a csv with reaches of interest:

In [6]:
# Load the CSV file with column 'Node_ID' with nodes of interest
# df = pd.read_csv('/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/PTs/SWOTCalVal_YR_node_id.csv') # PT nodes SWORD v16
# df = pd.read_csv('/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/PTs/SWOTCalVal_YR_KEY_20240704_20240826_v17b.csv') # PT nodes SWORD v17b
df = pd.read_csv('/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/YR_domain_v17b.csv') # GNSS nodes SWORD v17b & v16

# Remove duplicate reach_id values and convert to string list

# v17b field is 'v17b_reach_id'
reach_ids = df.drop_duplicates(subset=['v17b_reach_id'])['v17b_reach_id'].astype(str).tolist()
# v16 field is 'v16_reach_id'
# reach_ids = df.drop_duplicates(subset=['v16_reach_id'])['v16_reach_id'].astype(str).tolist()

print(reach_ids)

['81250800961', '81250800971', '81250800981', '81250800991', '81250800011', '81250800021', '81250800031', '81250800041', '81250800051', '81250700071', '81250700081', '81250900011', '81250900021', '81250900031', '81260300191', '81260300201', '81260300211', '81260300221', '81260300011', '81260300021', '81260300031', '81260300041', '81260300051', '81260300181', '81260300061', '81260300071', '81260300081', '81260300091', '81260300101', '81260300111', '81260300121', '81260300231', '81260300241', '81260300131', '81260300141', '81260300151', '81260300161', '81260300171', '81260400021', '81260400031', '81260400041', '81260400011', '81260500011', '81260500021', '81260500031', '81260500041', '81260500051', '81260500061', '81260500071', '81260500081', '81260500091', '81260500101', '81260500111', '81270100011', '81270100021', '81270100031', '81270100041', '81270100051', '81270100061', '81270100071', '81270100081', '81270100091', '81270100101', '81270100111', '81270100121', '81270100131', '81270100

In [7]:
# SWOT RiverTile folder and shapefile list

# Change directory based on SWORD version
# swot_rivertile_folder = "/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/SWOT/reach/RiverTile_v16"
swot_rivertile_folder = "/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/SWOT/reach/RiverTile_v17b"

SWOTshapefiles = [f for f in os.listdir(swot_rivertile_folder) if f.startswith("SWOT_L2_HR_RiverTile_Reach") and f.endswith(".shp")]
print(len(SWOTshapefiles))

# Set project CRS (EPSG:32606 for Yukon Flats)
YR_crs = "epsg:32606"

307


Field versions currently selected to pull:
reach_id,time,time_tai,time_str,p_lat,p_lon,wse,wse_u,wse_r_u,slope,slope_u,slope_r_u,width,width_u,area_total,area_tot_u,area_detct,
area_det_u,area_wse,layovr_val,node_dist,xtrk_dist,reach_q,reach_q_b,dark_frac,n_good_nod,p_n_nodes,partial_f,xovr_cal_q,p_dist_out

In [9]:
# Output CSV for node timeseries
summary_csv_path = os.path.join(swot_rivertile_folder, "RiverTile_domain_reach_timeseries_v17b.csv") #change filename as needed here!
fieldnames = [
    'SWOTFileName', 'reach_id', 'time', 'time_tai', 'time_str',
    'p_lat', 'p_lon', 'wse', 'wse_u', 'wse_r_u', 'slope', 'slope_u',' slope_r_u', 
    'width', 'width_u', 'area_total', 'area_tot_u', 'area_detct', 'area_det_u', 'area_wse',
    'layovr_val', 'node_dist', 'xtrk_dist', 'reach_q', 'reach_q_b',
    'dark_frac', 'n_good_nod', 'p_n_nodes', 'partial_f', 'xovr_cal_q', 'p_dist_out',
]
# need to add additional fields in csv here and at the bottom of the loop in next chunk

In [10]:
# iterate through all SWOT shapefiles to pull fields for each node of interest and save to output csv
with open(summary_csv_path, mode='w', newline='') as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()

    # Iterate through all SWOT shapefiles
    for SWOTshapefile in SWOTshapefiles:
        # Extract filename info
        SWOTfilename = os.path.basename(SWOTshapefile)[26:70] #gets cycle ID, tile ID, and start/end datetime

        # Load the SWOT shapefile & set the CRS
        shapefile_path = os.path.join(swot_rivertile_folder, SWOTshapefile)
        swot_rivertile = gpd.read_file(shapefile_path).to_crs(YR_crs)

        # Just process rows matching Node_IDs of interest
        swot_rivertile['reach_id'] = swot_rivertile['reach_id'].astype(str)
        matching_nodes = swot_rivertile[swot_rivertile['reach_id'].isin(reach_ids)]

        # pull fields of interest to write to csv
        for _, row in matching_nodes.iterrows():
            print(f"Processing {SWOTfilename} | reach_id {row['reach_id']}")

            writer.writerow({
                'SWOTFileName': SWOTfilename,
                'reach_id': row['reach_id'],
                'time': row['time'],
                'time_tai': row['time_tai'],
                'time_str': row['time_str'],
                'p_lat': row['p_lat'],
                'p_lon': row['p_lon'],
                'wse': row['wse'],
                'wse_u': row['wse_u'],
                'wse_r_u': row['wse_r_u'],
                'slope': row['slope'], 
                'slope_u': row['slope_u'],
                ' slope_r_u': row['slope_r_u'],
                'width': row['width'],
                'width_u': row['width_u'],
                'area_total': row['area_total'],
                'area_tot_u': row['area_tot_u'],
                'area_detct': row['area_detct'],
                'area_det_u': row['area_det_u'],
                'area_wse': row['area_wse'],
                'layovr_val': row['layovr_val'],
                'node_dist': row['node_dist'],
                'xtrk_dist': row['xtrk_dist'],
                'reach_q': row['reach_q'],
                'reach_q_b': row['reach_q_b'],
                'dark_frac': row['dark_frac'],
                'n_good_nod': row['n_good_nod'],
                'p_n_nodes': row['p_n_nodes'],
                'partial_f': row['partial_f'],
                'xovr_cal_q': row['xovr_cal_q'],
                'p_dist_out': row['p_dist_out'],
            })


Processing _020_302_034R_20240831T100122_20240831T10013 | reach_id 81260300101
Processing _020_302_034R_20240831T100122_20240831T10013 | reach_id 81260300111
Processing _020_302_034R_20240831T100122_20240831T10013 | reach_id 81260300121
Processing _020_302_034R_20240831T100122_20240831T10013 | reach_id 81260300131
Processing _020_302_034R_20240831T100122_20240831T10013 | reach_id 81260300141
Processing _020_302_034R_20240831T100122_20240831T10013 | reach_id 81260300151
Processing _020_302_034R_20240831T100122_20240831T10013 | reach_id 81260300161
Processing _020_302_034R_20240831T100122_20240831T10013 | reach_id 81260300171
Processing _020_302_034R_20240831T100122_20240831T10013 | reach_id 81260300231
Processing _020_302_034R_20240831T100122_20240831T10013 | reach_id 81260300241
Processing _020_302_034R_20240831T100122_20240831T10013 | reach_id 81260400011
Processing _020_302_034R_20240831T100122_20240831T10013 | reach_id 81260400021
Processing _020_302_034R_20240831T100122_20240831T10